# Reproducing Li, Namburi, Olson et al. (2022) from dandiset 000689

**Paper.** Li H, Namburi P, Olson JM, et al. (2022). Neurotensin orchestrates valence assignment in
the amygdala. *Nature* 608, 586-592.
[doi:10.1038/s41586-022-04964-y](https://doi.org/10.1038/s41586-022-04964-y). The claims file was
written from the accepted author manuscript in PubMed Central (PMC9583860): main text, Methods
with the per-panel statistics, and the legends of Figures 1 to 5 and Extended Data Figures 1 to 10.

**Dandiset.** [DANDI:000689](https://dandiarchive.org/dandiset/000689/0.240530.1923), version
0.240530.1923. The paper cites no DANDI version (it predates the dandiset, created in October 2023).
There are two published versions, 0.240524.2012 and 0.240530.1923; the data check below shows that
both list the same 85 assets with the same asset identifiers, so they do not differ in content.
The most recent, 0.240530.1923, is pinned, as the protocol asks when the paper cites none.

This notebook tests the electrophysiology, behavior and NT sensor photometry claims that the files
can support, and shows why the others are blocked.

**What the dandiset holds.** 31 sessions of BLA recordings during the three-cue discrimination
task (one per mouse, spike times of 1902 units, trial events, DeepLabCut pose in 29 of them), two
pose-only files for other sessions of two of those mice, 32 videos, and 20 photometry sessions of the
GRAB-NTS1.0 NT sensor in the BLA (5 mice, early and late acquisition of tone-sucrose and
tone-airpuff learning). No file says which mice are CRISPR and which are controls, nor which
projection (NAc or CeA) was tagged in each mouse, the units are not the paper's sorted single
units, and the pose has no timestamps, so its clock is fitted to the trial table (data check).

| Claim | Tested here | Outcome basis |
|---|---|---|
| fig3k-nt-sensor-sucrose-increase | yes | NT sensor files |
| fig3l-nt-sensor-airpuff-decrease | yes | NT sensor files |
| fig3-nt-sensor-early-dip | yes | NT sensor files |
| fig4-recording-count | yes | units table |
| fig4c-photoidentification | yes, total count only | light events and units; no projection labels |
| fig4d-seven-clusters | run, blocked, methods_underspecified | units table |
| fig5dh-behavior-clusters | yes | pose, on a clock fitted to the port entries |
| fig2c, ed8d, ed8fg, fig4e, ed8i, fig4fg (2), fig5a, fig5b, fig5c, ed8e, fig5fi, ed8os | blocked, metadata_insufficient | no group or projection labels |
| fig1c, ed1c, ed2, ed3, ed5f, fig1g, fig1h, fig2d, fig2g, fig2h, fig3bc, fig3e, fig3f, ed6, fig3h, ed9, ed9kl, ed10b, ed10 | blocked, data_missing | experiments not in the dandiset |


In [ ]:
import time, json, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import nt_bla as nb

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
started = time.time()
A = nb.assets()
print(f"dandiset {nb.DANDISET} version {nb.VERSION}: {len(A)} assets")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
# The two published versions: same assets?
other = []
url = f"{nb.API}/dandisets/{nb.DANDISET}/versions/0.240524.2012/assets/?page_size=200"
while url:
    r = requests.get(url, timeout=60).json(); other += r["results"]; url = r.get("next")
a1 = {a["path"]: (a["asset_id"], a["size"]) for a in A}
a2 = {a["path"]: (a["asset_id"], a["size"]) for a in other}
print("assets in 0.240530.1923:", len(a1), " in 0.240524.2012:", len(a2),
      " same paths:", set(a1) == set(a2), " same asset ids and sizes:", a1 == a2)
kind = lambda p: ("photometry nwb" if "ophys" in p else "ecephys nwb" if "ecephys" in p and p.endswith(".nwb")
                  else "pose-only nwb" if p.endswith(".nwb") else "video")
inv = pd.DataFrame({"path": list(a1), "GB": [a1[p][1] / 1e9 for p in a1]})
inv["kind"] = inv.path.map(kind)
inv["subject"] = inv.path.str.split("/").str[0]
print(inv.groupby("kind").agg(files=("path", "size"), subjects=("subject", "nunique"), GB=("GB", "sum")).round(1).to_string())

The two published versions hold the same assets, so the choice between them cannot change a result.
The archive's summary counts 36 subjects: 31 electrophysiology mice and 5 photometry mice.

In [ ]:
# Electrophysiology: spike times and trial tables of the 31 sessions (raw voltage is not read).
EP = sorted(a["path"] for a in A if a["path"].endswith(".nwb") and "ecephys" in a["path"])
t = time.time()
with ThreadPoolExecutor(4) as ex:
    E = list(ex.map(nb.load_ephys, EP))
print(f"loaded {len(E)} sessions in {time.time() - t:.0f} s")
rows = []
for e in E:
    tr = e["trials"]
    r = {"subject": e["subject"], "session": e["session_id"][-22:], "genotype": e["genotype"],
         "units": len(e["spikes"]), "channels": e["n_channels"], "dur_s": round(e["duration"])}
    for k, v in nb.CS_TYPES.items():
        r[k] = len(tr[v][0]) if v in tr else 0
    r["sucrose_deliv"] = len(tr.get("reward_delivery", ([],))[0])
    r["shock_relay"] = len(tr.get("shock_relay", ([],))[0])
    r["light"] = len(tr.get("phototagging", ([],))[0])
    cs = np.concatenate([tr[v][0] for v in nb.CS_TYPES.values() if v in tr])
    r["task_end_s"] = round(cs.max()); r["light_start_s"] = round(tr["phototagging"][0].min())
    rows.append(r)
ses = pd.DataFrame(rows)
print(ses.to_string(index=False))
print("\nmice:", ses.subject.nunique(), " sessions:", len(ses), " units:", ses.units.sum(),
      " genotype values:", sorted(ses.genotype.unique()))
print("sessions without neutral CS trials:", list(ses.subject[ses.neutral == 0]))
print("CS counts (sucrose, neutral, shock) in the other sessions:",
      ses[ses.neutral > 0][["sucrose", "neutral", "shock"]].apply(tuple, axis=1).value_counts().to_dict())

**Checked against the paper.** The session description says sucrose, neutral and shock CSs at 2:1:1,
and 25 sessions hold 60, 30 and 30 CSs (some with one or two fewer), as the Methods give (120
trials). Six sessions (H09 to H14, recorded on 2019-08-30, "Disc4") hold 75 sucrose and 45 shock CSs
and no neutral CS: the description does not fit them. Every file gives the genotype as "Wild type";
none says whether the mouse is in the CRISPR or the control group, nor whether Cav2-Cre was put in
the NAc or the CeA, which the group comparisons and the phototagging need. There are 31 mice where
the paper reports 30. H28 and H32 also have a pose-only file for a later discrimination day (Disc4,
Disc6) than their recording (Disc3, Disc5), so their recorded session may not be the last day,
which is the only day the paper analyses. Each session holds about 850 light events (the phototagging
sessions); no "fake" session is labelled.

In [ ]:
# Are the units sorted single units? Refractory violations and spikes shared between units.
def coinc(a, b, w=1e-4):
    if len(a) == 0 or len(b) < 2: return 0.0
    j = np.clip(np.searchsorted(b, a), 1, len(b) - 1)
    return float(np.mean(np.minimum(np.abs(b[j] - a), np.abs(b[j - 1] - a)) < w))
q = []
for e in E:
    sp = e["spikes"]
    isi = [np.mean(np.diff(s) < 0.001) if len(s) > 1 else np.nan for s in sp]
    allsp = np.concatenate(sp)
    shared = 1 - len(np.unique(np.round(allsp * 30000).astype(np.int64))) / len(allsp)
    nn = min(len(sp), 16)
    pair = [coinc(sp[i], sp[j]) for i in range(nn) for j in range(nn) if i != j]
    q.append({"subject": e["subject"], "units": len(sp), "median_frac_isi_lt_1ms": np.nanmedian(isi),
              "units_with_isi_viol_gt_5pct": int(np.sum(np.array(isi) > 0.05)),
              "frac_spike_samples_shared": shared, "median_pair_coincidence_0.1ms": np.median(pair)})
q = pd.DataFrame(q)
print(q.round(3).to_string(index=False))
print("\nover all units: median fraction of ISIs below 1 ms = %.3f; units with more than 5%% = %d of %d"
      % (q.median_frac_isi_lt_1ms.median(), q.units_with_isi_viol_gt_5pct.sum(), q.units.sum()))
print("units per session: %s" % q.units.value_counts().sort_index().to_dict())

**Checked against the paper.** The Methods say single units were sorted in Offline Sorter, and the
paper analyses 685 neurons from 30 mice (about 23 per mouse). The files hold 1902 units, 64 in 21
of 31 sessions (two per channel of the 32-channel optrode). In 1061 units more than 5% of the
inter-spike intervals are below 1 ms (a median of 17 to 35% per session in H01 to H14), which a
well-isolated single neuron does not show, and in each session 6 to 65% of spike samples recur in
another unit. These are not the paper's sorted single units:
they look like unsorted or multi-unit threshold crossings. The claims that use units are tested on
them as they are, and this is recorded as a finding.

In [ ]:
# Every text field of each recording file, the units and electrodes columns, and the archive's
# per-asset metadata: is there any group (CRISPR, control) or projection (NAc, CeA) label?
import re
def text_fields(path):
    h = nb.open_h5(path)
    out = {}
    def visit(name, obj):
        if isinstance(obj, nb.h5py.Dataset) and obj.dtype.kind in "OSU" and obj.size <= 64 and not name.startswith(("units/", "intervals/", "acquisition/", "processing/")):
            v = obj[()]
            out[name] = " | ".join(str(nb._s(x)) for x in np.atleast_1d(v) if isinstance(nb._s(x), str))
    h["general"].visititems(lambda n, o: visit("general/" + n, o))
    out["session_description"] = nb._s(h["session_description"][()])
    out["units columns"] = ", ".join(sorted(h["units"].keys()))
    out["electrodes columns"] = ", ".join(sorted(h["general/extracellular_ephys/electrodes"].keys()))
    out["electrode locations"] = ", ".join(sorted(set(nb._s(x) for x in h["general/extracellular_ephys/electrodes/location"][:])))
    return out
with ThreadPoolExecutor(4) as ex:
    TF = list(ex.map(text_fields, EP))
fields = sorted(set().union(*TF))
for f in fields:
    vals = sorted(set(t.get(f, "(absent)") for t in TF))
    shown = vals if f != "general/experiment_description" else [v[:80] + "..." for v in vals]
    print(f"{f}: {len(vals)} distinct value(s): {shown[:6]}{' ...' if len(vals) > 6 else ''}")
pat = re.compile(r"crispr|control|scrambl|sgrna|guide|nac|accumbens|cea|central amyg|cav2|projection|group", re.I)
hits = sorted({(f, m.group(0)) for t in TF for f, v in t.items() if f != "general/experiment_description" for m in pat.finditer(v)})
print("\nlabel words in the files' fields (experiment_description, the paper's abstract, left out):", hits)
am = []
for a in A:
    if a["path"] in EP:
        r = requests.get(f"{nb.API}/dandisets/{nb.DANDISET}/versions/{nb.VERSION}/assets/{a['asset_id']}/", timeout=60).json()
        am.append({k: str(v) for k, v in r.get("wasAttributedTo", [{}])[0].items() if k not in ("identifier",)})
am = pd.DataFrame(am)
print("\narchive asset metadata, subject fields:", list(am.columns))
print({c: sorted(am[c].unique().tolist())[:4] for c in am.columns if c not in ("age",)})


No field of the files, and no subject field of the archive's metadata, names a CRISPR or control
group, a guide RNA, or the NAc or CeA (the words found, if any, are listed above with their field).
The units table holds only spike times and a unit name; the electrodes table gives no location.

In [ ]:
# Clock check for the spikes and the trial table: CS-evoked change in population rate at the
# true CS onsets against the same onsets shifted 10 s earlier (inside the inter-trial interval).
def pop_resp(e, shift):
    on = np.concatenate([e["trials"][v][0] for v in nb.CS_TYPES.values() if v in e["trials"]]) + shift
    out = []
    for sp in e["spikes"]:
        p = nb.psth(sp, on, 1.0, 1.0, 0.1)
        b = p[:10].mean()
        out.append((p[10:13].mean() - b) / (p[:10].std(ddof=1) + 1e-9))
    return np.median(np.abs(out))
ck = pd.DataFrame([{"subject": e["subject"], "true_onsets": pop_resp(e, 0.0), "shifted_-10s": pop_resp(e, -10.0)} for e in E])
print("median |z| of the 0 to 0.3 s response across units, per session:")
print(ck.round(2).to_string(index=False))
print("sessions where the true onsets give the larger response: %d of %d" % ((ck.true_onsets > ck["shifted_-10s"]).sum(), len(ck)))

The units respond more at the true CS onsets than at onsets shifted into the inter-trial interval in
the sessions counted above, which is consistent with the spikes and the trial table sharing a clock
(a check that supports, without proving, the alignment). It fails in H17, whose spike times end
at 249 s of a 12118 s recording (next cell): its units cannot be aligned with its trials. In H22,
H27, H34 and H39 the two values are close, so the alignment is not shown there.

In [ ]:
# Span of the spike times against the length of the recording, per session.
span = pd.DataFrame([{"subject": e["subject"], "first_spike_s": round(min(s[0] for s in e["spikes"] if len(s)), 1),
                      "last_spike_s": round(max(s[-1] for s in e["spikes"] if len(s)), 1), "recording_s": round(e["duration"], 1),
                      "task_start_s": round(min(e["trials"][v][0].min() for v in nb.CS_TYPES.values() if v in e["trials"]), 1),
                      "task_end_s": round(max(e["trials"][v][0].max() for v in nb.CS_TYPES.values() if v in e["trials"]), 1),
                      "light_before_task": int(np.sum(e["trials"]["phototagging"][0] < min(e["trials"][v][0].min() for v in nb.CS_TYPES.values() if v in e["trials"]))),
                      "light_after_task": int(np.sum(e["trials"]["phototagging"][0] > max(e["trials"][v][0].max() for v in nb.CS_TYPES.values() if v in e["trials"])))} for e in E])
print(span.to_string(index=False))
print("sessions whose spikes stop before 90% of the recording:", list(span.subject[span.last_spike_s < 0.9 * span.recording_s]))
print("sessions with light events before the task:", list(span.subject[span.light_before_task > 0]))
# Light events form two blocks per session: split at the largest gap between events.
for e in E[:3] + E[-3:]:
    x = e["trials"]["phototagging"][0]; g = np.argmax(np.diff(x))
    print(f'{e["subject"]}: blocks of {g + 1} and {len(x) - g - 1} light events, gap {np.diff(x)[g]:.0f} s')

H17's spike times all fall within the first 249 s of a recording of 12118 s, while its trial events
cover the whole session: its spike times are not on the trial clock (a file error). Every session
holds about 854 light events in two blocks of about 427, which fits the Methods' two light sessions
(a "fake" one with the patch cord disconnected, then the real one), but no label says which block
is the fake session. In 12 sessions (H31 to H42) one block comes before the task and one after it;
the Methods put both after the task.

In [ ]:
# Does the pose share the trial clock? The pose series give only a start time (0 s) and a nominal
# 15 frames/s, no timestamps. At each port entry (beam break, trial table) the head should be at the
# port. (1) With the nominal clock, the fixed lag (pose time minus trial time) that minimises the
# median head-to-port distance at port entries, in each third of the session. (2) A linear map,
# pose time = T + offset + factor * (t - T), fitted to all port entries (nb.fit_pose_clock), and
# the best extra lag left in each third under it (0 when the map holds throughout).
def pose_clock(p):
    h = nb.open_h5(p)
    subj = p.split("/")[0][4:]
    if "processing/behavior/PoseEstimation" not in h:
        return subj, None
    pose = nb.load_pose(h); r, t0 = pose["rate"], pose["t0"]
    m = nb.pose_measures(pose)
    dist = m["distance_to_port"]
    tt = h["intervals/trials/trial_type"][:].astype(str); st = h["intervals/trials/start_time"][:]
    pe = np.sort(st[tt == "port_entry"])
    out = {"subject": subj, "pose_s": round(len(dist) / r), "port_entries": len(pe)}
    lags = np.arange(-200, 200.01, 0.2)
    for k, seg in enumerate(np.array_split(pe, 3)):
        med = nb._median_at(dist, r, t0, seg[None, :] + lags[:, None])
        out[f"lag{k + 1}_s"] = round(float(lags[np.nanargmin(med)]), 1)
    c = nb.fit_pose_clock(dist, r, t0, pe)
    out.update({"fit_offset_s": round(c["offset"], 1), "fit_factor": round(c["factor"], 4),
                "resid1_s": c["resid_lag_thirds"][0], "resid2_s": c["resid_lag_thirds"][1], "resid3_s": c["resid_lag_thirds"][2],
                "dist_fit": round(c["med_dist_fit"], 1), "dist_lag0": round(c["med_dist_lag0"], 1), "dist_all_frames": round(c["med_dist_all"], 1)})
    return subj, (out, {"measures": m, "rate": r, "t0": t0, "clock": c})
t = time.time()
with ThreadPoolExecutor(4) as ex:
    PC = dict(ex.map(pose_clock, EP))
pl = pd.DataFrame([v[0] for v in PC.values() if v is not None])
print(f"pose read and clocks fitted in {time.time() - t:.0f} s; sessions without pose: {[k for k, v in PC.items() if v is None]}")
print(pl.round(1).assign(fit_factor=pl.fit_factor.round(4)).to_string(index=False))
spread = pl[["lag1_s", "lag2_s", "lag3_s"]].max(axis=1) - pl[["lag1_s", "lag2_s", "lag3_s"]].min(axis=1)
resid = pl[["resid1_s", "resid2_s", "resid3_s"]].abs().max(axis=1)
pl["aligned"] = resid <= 1.0
print("\nnominal clock: median |lag| %.1f s; sessions whose best lag changes by more than 5 s within the session: %d of %d"
      % (np.median(np.abs(pl[["lag1_s", "lag2_s", "lag3_s"]].values)), int((spread > 5).sum()), len(pl)))
print("linear map: sessions with every third within 1 s of the map: %d of %d; not: %s"
      % (pl.aligned.sum(), len(pl), list(pl.subject[~pl.aligned])))
print("fitted factor in the aligned sessions: median %.4f, range %.4f to %.4f; median distance at port entries %.1f px under the map, %.1f px at lag 0, %.1f px over all frames"
      % (pl.fit_factor[pl.aligned].median(), pl.fit_factor[pl.aligned].min(), pl.fit_factor[pl.aligned].max(),
         pl.dist_fit[pl.aligned].median(), pl.dist_lag0[pl.aligned].median(), pl.dist_all_frames[pl.aligned].median()))
print("aligned sessions with a factor between 0.987 and 0.989: %d of %d" % (pl.fit_factor[pl.aligned].between(0.987, 0.989).sum(), pl.aligned.sum()))


In [ ]:
# The video assets: frame count, frame rate and duration from each file's container header
# (streamed, no frame is decoded), against the pose series and the electrophysiology recording.
import av
def video_info(a):
    out = {"subject": a["path"].split("/")[0][4:], "file": a["path"].split("/")[1][:40], "container": a["path"][-3:]}
    try:
        rf = nb.RangeFile(f"{nb.API}/assets/{a['asset_id']}/download/")
        f = av.open(rf)
        v = f.streams.video[0]
        out.update({"frames": v.frames, "rate": float(v.average_rate), "duration_s": round(f.duration / 1e6, 1) if f.duration else np.nan,
                    "MB_read": round(rf.fetched / 1e6, 1)})
        f.close()
    except Exception as ex:
        out["error"] = str(ex)[:60]
    return out
VID = [a for a in A if not a["path"].endswith(".nwb")]
with ThreadPoolExecutor(8) as ex:
    vi = pd.DataFrame(list(ex.map(video_info, VID)))
rec = {e["subject"]: e["duration"] for e in E}
vi["recording_s"] = vi.subject.map(rec).round(1)
vi["pose_s"] = vi.subject.map(dict(zip(pl.subject, pl.pose_s)))
print(vi.sort_values("subject").to_string(index=False))
print("mkv files report no frame count in their header (0); their rate is the container's nominal one.")


With the nominal clock (15 frames per second from 0 s) the head is closest to the port tens of
seconds away from the port entries (median 80 s), and the best lag drifts within the session in 28
of 29 sessions (columns lag1 to lag3). A linear map with an offset and a rate factor removes the
drift in 21 of the 29 sessions: under it, the best extra lag in each third is within 1 s of zero,
20 of those 21 fitted factors lie between 0.9874 and 0.9885 (H27 is at 1.0000), and the median
head-to-port distance at port entries is 38 pixels against 105 over all frames. A common factor of
about 0.988 is what a video whose true frame rate is about 1.2% below the nominal 15 per second
would give. In H11, H14, H17, H23, H25, H29, H39 and H40 the map leaves a lag of 1 to 15 s in some
third, or the distance stays near that over all frames (H29, H39, H40).

The video headers (next output) give a constant 15 frames per second for the mp4 files and
durations equal to the pose series, except H07, H09 and H10, whose videos run 800 to 1500 s longer
than their pose; the mkv files give no frame count. So the drift is not visible in the files' own
metadata; the original videos were .asf files (PoseEstimation original_videos), whose timing is
lost. The pose can be put on the trial clock in the 21 aligned sessions, by a map estimated from the
data, not given by the files. This is an assumption for the behavior claim, which uses only those
sessions. The map is consistent with the data, not shown by them.

In [ ]:
# Photometry: the 20 NT sensor sessions.
PP = sorted(a["path"] for a in A if "ophys" in a["path"])
t = time.time()
with ThreadPoolExecutor(4) as ex:
    S = list(ex.map(nb.load_photometry, PP))
print(f"loaded {len(S)} sessions in {time.time() - t:.0f} s")
ph = []
for s in S:
    sid = s["session_id"]
    lick_rel = np.concatenate([s["licks"][(s["licks"] > c - 2) & (s["licks"] < c + 6)] - c for c in s["cs_on"]])
    ph.append({"subject": s["subject"], "task": "sucrose" if "Reward" in sid else "airpuff",
               "phase": "early" if "early" in sid else "late", "fluorophore": s["fluorophore"][:14],
               "fiber": s["fiber_location"], "nm": s["wavelengths"], "frames": s["n_frames"],
               "dt_ms": tuple(round(x * 1000, 1) for x in s["frame_dt"]),
               "phase_means": tuple(round(x, 3) for x in s["phase_means"]),
               "ph415": s["phase415"], "ph470": s["phase470"],
               "agree": round(min(s["phase_agree415"], s["phase_agree470"]), 3),
               "labels_moved": s["labels_moved415"] + s["labels_moved470"],
               "phase_means_after_irregular": tuple(round(x, 3) for x in s["phase_means_after_irregular"]),
               "cues": len(s["cs_on"]), "cue_s": round(float(np.median(s["cs_dur"])), 2),
               "min_gap_s": round(float(np.diff(s["cs_on"]).min()), 1),
               "lick_frames_-2_0": int(((lick_rel >= -2) & (lick_rel < 0)).sum()),
               "lick_frames_2_4": int(((lick_rel >= 2) & (lick_rel < 4)).sum())})
ph = pd.DataFrame(ph)
print(ph.to_string(index=False))

**Checked against the paper.** The files hold the GRAB-NTS1.0 sensor ("neurotensin-fluorescent
sensor") in the BLA of five wild-type mice, with 415 and 470 nm excitation, which is the experiment
of Figure 3j-l (N = 5; the Methods exclude one of six mice for fiber placement, and the subjects are
Mouse1, 2, 4, 5 and 6). Frames come every 16.7 ms (60 Hz), not at the Methods' 40 Hz: one series
interleaves three frame types. The LED labels mark every 415 nm frame and every 470 nm frame (a third
of the frames each); these sit on a fixed phase of three ("agree" is the fraction of labelled frames
on the most common phase). In Mouse6's late sucrose session (M-5) the labels move by one frame after
a 2.3 ms frame interval ("labels_moved"), while the data keep their order: the mean of each phase
after that interval ("phase_means_after_irregular") is the same as over the session, with the
near-constant third frame type still on phase 0. Following the labels there would read that constant
frame type as the 470 nm channel, so the channels are taken from the frame order, at the phase the
labels give before the first irregular interval.
The third frame type is a near-constant value around 0.13 to 0.20 that no metadata describes. Each session holds about
60 cues lasting 1.9 to 2 s, about 30 s apart. The exception is Mouse1's early airpuff session (M-0):
it holds 29 cues, and its last three come 4.7 and 7.4 s apart, one of them a 4.7 s bout, which
looks like a disrupted end of the session. These cues are among the "all trials" readings; the
"first 16 trials" readings do not reach them. The Methods
give a 4 s CS with the US 2 s after onset and about half of the CSs unpaired; the files mark one
kind of cue, 2 s long, and no US. In sucrose sessions licking rises after the cue ends (frames with
a lick 2 to 4 s after onset against the 2 s before), which places the sucrose at the end of the 2 s
cue, as the Methods' US time of 2 s gives; no file marks whether a cue was paired or unpaired, nor
the airpuff. All cues are therefore analysed as CS-US trials, an assumption recorded with the
claims.

In [ ]:
# Per-trial sucrose check: in the late sucrose sessions, how many cues are followed by licking
# within 2 to 5 s of onset (a paired trial should be, an unpaired one need not)?
for s in S:
    if "Reward" in s["session_id"]:
        n = [np.sum((s["licks"] >= c + 2) & (s["licks"] < c + 5)) > 0 for c in s["cs_on"]]
        print(f'{s["session_id"]:34s} cues followed by a lick within 2-5 s: {sum(n)} of {len(n)}')

In the late sucrose sessions most cues are followed by licking, so the files' cues are mostly, and
perhaps all, paired with sucrose; the early sessions, before learning, cannot tell.

## Photometry claims (Figure 3j-l)

Processing as the claims' criterion gives it (Methods, Fiber photometry, Data analysis): 415 and
470 nm channels median-filtered over 200 ms and averaged into 200 ms (5 Hz) bins from -5 to 10 s
around each CS onset; the trial-averaged 470 nm trace regressed on the trial-averaged 415 nm trace
over -5 to 0 s (the 415 nm fit subtracted from the 470 nm trace), the residuals z-scored against -5 to 0 s. Windows: CS 0 to 2 s, US 2 to 7 s (Figure 3
legend). Early and late are the files' early and late acquisition sessions. Readings run, as the
criterion lists them: (a) first 16 trials or (b) all trials; (i) area under the z-scored curve or
(ii) mean z. One more detail the Methods leave open is run both ways: the z-score uses each trial's
own baseline ("trial") or the session's pooled baseline ("session").
The regression is fitted on the trial-averaged baseline, as the Methods say; the fitted slope,
intercept and correlation of each session are printed, and a fit on the baseline bins of every
trial pooled is run as a variant, since the averaged baseline may hold too little variance to fit.


In [ ]:
TRIALS, FITS = {}, []
for s in S:
    key = (s["subject"], "sucrose" if "Reward" in s["session_id"] else "airpuff", "early" if "early" in s["session_id"] else "late")
    for zm in ("trial", "session"):
        tb, Z, keep, fit = nb.photometry_trials(s, zmode=zm, return_fit=True)
        TRIALS[key + (zm,)] = (tb, Z)
    tb, Z, keep, fitp = nb.photometry_trials(s, zmode="trial", fit="pooled", return_fit=True)
    TRIALS[key + ("trial, pooled fit",)] = (tb, Z)
    FITS.append({"mouse": key[0], "task": key[1], "phase": key[2], "slope": fit[0], "intercept": fit[1], "r": fit[2],
                 "pooled_slope": fitp[0], "pooled_intercept": fitp[1], "pooled_r": fitp[2]})
print("470 nm on 415 nm regression over -5 to 0 s, per session (trial-averaged fit, and pooled-trials variant):")
print(pd.DataFrame(FITS).round(3).to_string(index=False))
print("trials kept per session (all cues have a full -5 to 10 s window):",
      {f"{k[0]} {k[1]} {k[2]}": v[1].shape[0] for k, v in TRIALS.items() if k[3] == "trial"})
mice = sorted({k[0] for k in TRIALS})

def measure(task, phase, zm, sub, meas, lo, hi):
    out = []
    for m in mice:
        tb, Z = TRIALS[(m, task, phase, zm)]
        Z = Z[:16] if sub == "first16" else Z
        out.append(nb.window_measure(tb, Z, lo, hi, meas).mean())
    return np.array(out)

def compare(task, zmodes=("trial", "session")):
    rows = []
    for zm in zmodes:
        for sub in ("first16", "all"):
            for meas in ("auc", "mean"):
                r = {"zscore": zm, "trials": sub, "measure": meas}
                for win, (lo, hi) in (("US", (2, 7)), ("CS", (0, 2))):
                    e, l = measure(task, "early", zm, sub, meas, lo, hi), measure(task, "late", zm, sub, meas, lo, hi)
                    tt = stats.ttest_rel(l, e)
                    d = l - e
                    ci = stats.t.interval(0.95, len(d) - 1, d.mean(), stats.sem(d))
                    r.update({f"{win}_late-early": d.mean(), f"{win}_ci_low": ci[0], f"{win}_ci_high": ci[1],
                              f"{win}_t": tt.statistic, f"{win}_p": tt.pvalue, f"{win}_mice_in_direction": None})
                    r[f"{win}_mice_up"] = int((d > 0).sum())
                rows.append(r)
    return pd.DataFrame(rows).drop(columns=["US_mice_in_direction", "CS_mice_in_direction"])

### fig3k-nt-sensor-sucrose-increase

> The BLA NT sensor response to sucrose delivery increases after reward learning.
> (Paper: US window late against early, paired t-test t(4) = 8.709, P = 0.001, effect 4.14, CI 2.744
> to 5.538; CS window t(4) = 1.788, P = 0.1483.) Reproduced when the US comparison has p < 0.05 with
> late higher under every reading.

In [ ]:
res_suc = compare("sucrose")
print(res_suc.round(4).to_string(index=False))
print("\nPer mouse, US window AUC (trial z-score, first 16 trials):")
print(pd.DataFrame({"mouse": mice, "early": measure("sucrose", "early", "trial", "first16", "auc", 2, 7),
                    "late": measure("sucrose", "late", "trial", "first16", "auc", 2, 7)}).round(2).to_string(index=False))
print("\nReadings with US p < 0.05 and late higher: %d of %d" % (((res_suc["US_p"] < 0.05) & (res_suc["US_late-early"] > 0)).sum(), len(res_suc)))
print("\nVariant, regression fitted on the pooled baseline bins of every trial (per-trial z-score):")
print(compare("sucrose", ("trial, pooled fit",))[["zscore", "trials", "measure", "US_late-early", "US_ci_low", "US_ci_high", "US_t", "US_p", "US_mice_up"]].round(4).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, task in zip(axes, ("sucrose", "airpuff")):
    for phase, col in (("early", "tab:blue"), ("late", "tab:red")):
        tr = np.array([TRIALS[(m, task, phase, "trial")][1][:16].mean(0) for m in mice])
        tb = TRIALS[(mice[0], task, phase, "trial")][0] + 0.1
        ax.plot(tb, tr.mean(0), color=col, label=phase)
        ax.fill_between(tb, tr.mean(0) - stats.sem(tr), tr.mean(0) + stats.sem(tr), color=col, alpha=0.2)
        print(f"{task} {phase}: plotted mean z from {tr.mean(0).min():.2f} to {tr.mean(0).max():.2f}")
    ax.axvspan(2, 7, color="0.9"); ax.axvline(0, color="k", lw=0.5)
    ax.set_title(f"NT sensor, {task} (first 16 trials, mean of 5 mice)"); ax.set_xlabel("s from CS onset")
axes[0].set_ylabel("z"); axes[0].legend(); plt.tight_layout(); plt.show()

**Result: partially reproduced.** Under the per-trial z-score the US response to sucrose is larger in
the late session under all four readings of the criterion. With all trials it is significant: AUC
+3.53 (95% CI 1.82 to 5.24), t(4) = 5.74, p = 0.0046, 5 of 5 mice higher, inside the paper's CI of
2.744 to 5.538. With the first 16 trials it is not: AUC +2.94, p = 0.093, 4 of 5 mice higher. So the
criterion's "p < 0.05 under every reading" holds in 2 of its 4 readings. Under the session-pooled
z-score no reading reaches p < 0.05 (p >= 0.16), and with the regression fitted on the pooled
baseline bins of every trial neither (p >= 0.44). The CS window does not change under any reading
(p >= 0.25), as in the paper. The per-trial z-score is the reading adopted (an assumption: the
Methods z-score "using data a baseline window of -5 s to 0 s", most simply read as each trial's own
window).

### fig3l-nt-sensor-airpuff-decrease

> The BLA NT sensor response to airpuff decreases after punishment learning.
> (Paper: US window t(4) = 3.996, P = 0.0162, effect -3.657, CI -6.083 to -0.9583; CS t(4) = 2.137,
> P = 0.0994.) Reproduced when the US comparison has p < 0.05 with late lower under every reading.

In [ ]:
res_air = compare("airpuff")
print(res_air.round(4).to_string(index=False))
print("\nPer mouse, US window AUC (trial z-score, first 16 trials):")
print(pd.DataFrame({"mouse": mice, "early": measure("airpuff", "early", "trial", "first16", "auc", 2, 7),
                    "late": measure("airpuff", "late", "trial", "first16", "auc", 2, 7)}).round(2).to_string(index=False))
print("\nReadings with US p < 0.05 and late lower: %d of %d" % (((res_air["US_p"] < 0.05) & (res_air["US_late-early"] < 0)).sum(), len(res_air)))
print("\nVariant, regression fitted on the pooled baseline bins of every trial (per-trial z-score):")
print(compare("airpuff", ("trial, pooled fit",))[["zscore", "trials", "measure", "US_late-early", "US_ci_low", "US_ci_high", "US_t", "US_p", "US_mice_up"]].round(4).to_string(index=False))


**Result: not reproduced.** Under the per-trial z-score the US response to airpuff is lower in the
late session under all four readings (AUC -5.47 with the first 16 trials, 5 of 5 mice lower, and
-4.66 with all trials; the paper's effect is -3.657), but no reading reaches p < 0.05 (smallest
p = 0.074, t(4) = -2.40). The session-pooled z-score gives p >= 0.096, and the pooled-baseline fit
p >= 0.23. The CS window does not change significantly under any reading (p >= 0.08).

### fig3-nt-sensor-early-dip

> Early in acquisition, the BLA NT sensor signal transiently decreases after both sucrose and
> airpuff. Reproduced when the mean z-scored signal over mice in the US window (2 to 7 s from CS
> onset) is below zero in early trials for both sucrose and airpuff.

In [ ]:
rows = []
for zm in ("trial", "session"):
    for sub in ("first16", "all"):
        for meas in ("mean", "auc"):
            r = {"zscore": zm, "trials": sub, "measure": meas}
            for task in ("sucrose", "airpuff"):
                v = measure(task, "early", zm, sub, meas, 2, 7)
                r[f"{task}_mean"] = v.mean(); r[f"{task}_mice_below_0"] = int((v < 0).sum())
                r[f"{task}_p_vs_0"] = stats.ttest_1samp(v, 0).pvalue
            rows.append(r)
dip = pd.DataFrame(rows)
print(dip.round(3).to_string(index=False))
print("\nReadings with both early means below zero: %d of %d" % (((dip.sucrose_mean < 0) & (dip.airpuff_mean < 0)).sum(), len(dip)))
print("\nEarly sessions, per mouse, mean z 2-7 s (trial z-score, first 16 trials):")
print(pd.DataFrame({"mouse": mice, "sucrose": measure("sucrose", "early", "trial", "first16", "mean", 2, 7),
                    "airpuff": measure("airpuff", "early", "trial", "first16", "mean", 2, 7)}).round(2).to_string(index=False))
# Variant for the word "transient" (asked of a reader): the lowest point, 2 to 7 s, of the trace
# averaged over trials and then over mice, and of each mouse's trial-averaged trace.
print("\nVariant: minimum of the early trace in 2 to 7 s (z), mean of 5 mice and per mouse:")
for zm in ("trial", "session"):
    for sub in ("first16", "all"):
        r = {}
        for task in ("sucrose", "airpuff"):
            tb = TRIALS[(mice[0], task, "early", zm)][0] + 0.1
            w = (tb > 2) & (tb < 7)
            tr = np.array([(TRIALS[(m, task, "early", zm)][1][:16] if sub == "first16" else TRIALS[(m, task, "early", zm)][1]).mean(0) for m in mice])
            r[task] = f"mean-trace min {tr.mean(0)[w].min():+.2f} at {tb[w][np.argmin(tr.mean(0)[w])]:.1f} s; per mouse {np.round(tr[:, w].min(1), 2).tolist()}"
        print(f"{zm:8s} {sub:8s} sucrose: {r['sucrose']}\n                  airpuff: {r['airpuff']}")


**Result: not reproduced.** In the early sessions the mean over mice of the US-window signal is
below zero for sucrose under the per-trial z-score (-0.62 with the first 16 trials, -0.40 with all
trials) but above zero for airpuff (+0.11 and +0.19; M-4 at +2.15), so the criterion, which needs
both below zero, is not met under any per-trial reading. Under the session-pooled z-score the signs
change with the trial subset (0 of 8 readings in all). No mean differs from zero (p >= 0.23). Read
as a brief dip, the paper's word "transient", the mouse-averaged trace does go below zero within 2 to
7 s for both outcomes under the per-trial z-score (lowest -1.09 for sucrose and -0.63 for airpuff,
first 16 trials); whether the criterion's window mean is the right measure for a transient decrease
is asked of a reader.

## Electrophysiology claims (Figure 4)

### fig4-recording-count

> 685 BLA neurons were recorded from 30 mice during the three-cue discrimination task. Units from
> the last discrimination-task session of each mouse, with baseline firing of at least 0.3 Hz
> (Methods). Whether 685 is counted before or after that exclusion is not said; both counts are
> shown and either may match (expected 685 +/- 68.5 neurons, 30 +/- 3 mice).

Baseline rate: the mean rate in the 1 s before each CS onset, over all CSs of the session (the PSTH
baseline of the Methods); the whole-session rate is shown as a variant.

In [ ]:
base_rate, sess_rate, unit_rows = {}, {}, []
for e in E:
    on = np.sort(np.concatenate([e["trials"][v][0] for v in nb.CS_TYPES.values() if v in e["trials"]]))
    for u, sp in enumerate(e["spikes"]):
        b = np.mean(np.searchsorted(sp, on) - np.searchsorted(sp, on - 1.0))
        unit_rows.append({"subject": e["subject"], "unit": u, "baseline_hz": b, "session_hz": len(sp) / e["duration"]})
U = pd.DataFrame(unit_rows)
n_all, n_base, n_sess = len(U), int((U.baseline_hz >= 0.3).sum()), int((U.session_hz >= 0.3).sum())
print(f"units before exclusion: {n_all}; with baseline >= 0.3 Hz: {n_base}; with session rate >= 0.3 Hz: {n_sess}")
print(f"mice: {U.subject.nunique()}")
print("per mouse (all units / kept by baseline):")
print(U.groupby("subject").agg(units=("unit", "size"), kept=("baseline_hz", lambda x: int((x >= 0.3).sum()))).T.to_string())
print(f"\nexpected 685 +/- 68.5 neurons: before exclusion {abs(n_all - 685) <= 68.5}, after {abs(n_base - 685) <= 68.5}")
print(f"expected 30 +/- 3 mice: {abs(U.subject.nunique() - 30) <= 3}")
print("units per mouse: median %.0f (paper 685 / 30 = 22.8)" % U.groupby("subject").size().median())
ratio = ses[["sucrose", "neutral", "shock"]]
near = (ratio.neutral > 0) & ((ratio.sucrose - 2 * ratio.neutral).abs() <= 2) & ((ratio.shock - ratio.neutral).abs() <= 2)
print(f"sessions with sucrose, neutral and shock CSs near 2:1:1 (within 2 trials): {int(near.sum())} of {len(ses)}; not: {list(ses.subject[~near])}")


**Result: partially reproduced.** The number of mice (31) is within the tolerance of 30. The unit
count is not: 1902 units, 1739 with a baseline rate of at least 0.3 Hz (1884 by whole-session rate),
against the paper's 685; neither is within 68.5 of 685. The units are not sorted single units (data
check), which fits a median of 64 per mouse instead of 23. The 2:1:1 ratio of the statement holds in
25 of 31 sessions; H09 to H14 have no neutral CS.

### fig4d-seven-clusters

> Unsupervised hierarchical clustering of BLA responses to the sucrose, neutral and shock CSs gives
> seven functional clusters. Per neuron, 20 ms PSTHs from -1 to 1 s around CS onset, z-scored with
> the 1 s baseline, smoothed with a Gaussian-weighted moving average (smoothing factor 0.85), the
> three CS traces concatenated; Ward linkage with correlation distance cut at 35% of the maximum
> linkage distance; readings (a) Ward on the correlation distance matrix, (b) Ward with Euclidean
> distance on centred, unit-norm rows. Reproduced when one reading gives seven clusters.

Units with a baseline rate below 0.3 Hz are excluded (Methods). The six sessions without neutral
CSs cannot give the concatenated trace and are left out; so are units whose baseline SD is zero.
The "smoothing factor of 0.85" sets the window through a MATLAB heuristic the paper does not give,
so the smoothing window is run over a range (in 20 ms bins, smoothing each CS trace separately).

In [ ]:
feats, owner = [], []
dropped = {"low_rate": 0, "no_neutral_session": 0, "zero_sd": 0}
for e in E:
    on = {k: e["trials"][v][0] for k, v in nb.CS_TYPES.items() if v in e["trials"]}
    allon = np.sort(np.concatenate(list(on.values())))
    for u, sp in enumerate(e["spikes"]):
        b = np.mean(np.searchsorted(sp, allon) - np.searchsorted(sp, allon - 1.0))
        if b < 0.3: dropped["low_rate"] += 1; continue
        if "neutral" not in on: dropped["no_neutral_session"] += 1; continue
        z = np.array([nb.zscore_psth(nb.psth(sp, on[k])) for k in ("sucrose", "neutral", "shock")])
        if np.isnan(z).any(): dropped["zero_sd"] += 1; continue
        feats.append(z); owner.append(e["subject"])
F = np.array(feats); owner = np.array(owner)
print("units clustered:", len(F), "from", len(set(owner)), "mice; dropped:", dropped)

def cluster(F, window):
    X = nb.gauss_smooth(F, window).reshape(len(F), -1)
    Za = linkage(pdist(X, "correlation"), "ward")
    ca = fcluster(Za, 0.35 * Za[:, 2].max(), "distance")
    Xc = X - X.mean(1, keepdims=True); Xc /= np.linalg.norm(Xc, axis=1, keepdims=True)
    Zb = linkage(Xc, "ward")
    cb = fcluster(Zb, 0.35 * Zb[:, 2].max(), "distance")
    return ca, cb

rows = []
CL = {}
for w in (1, 3, 5, 7, 9, 11, 15, 21, 31, 41, 51, 75, 101):
    ca, cb = cluster(F, w)
    CL[w] = (ca, cb)
    rows.append({"window_bins": w, "window_ms": w * 20, "clusters_a_corr_ward": ca.max(), "clusters_b_unitnorm_ward": cb.max(),
                 "sizes_a": sorted(np.bincount(ca)[1:].tolist(), reverse=True)})
cl = pd.DataFrame(rows)
print(cl.to_string(index=False))
print("\nwindows giving 7 clusters: reading (a)", list(cl.window_bins[cl.clusters_a_corr_ward == 7]),
      " reading (b)", list(cl.window_bins[cl.clusters_b_unitnorm_ward == 7]))

In [ ]:
# Cut-off sensitivity for one window (11 bins = 220 ms): clusters as a function of the cut fraction.
Xw = nb.gauss_smooth(F, 11).reshape(len(F), -1)
Za = linkage(pdist(Xw, "correlation"), "ward")
print("reading (a), window 11 bins: clusters by cut fraction of the maximum linkage distance")
print({f: int(fcluster(Za, f * Za[:, 2].max(), "distance").max()) for f in (0.25, 0.3, 0.35, 0.4, 0.45, 0.5)})
# Per mouse composition of the clusters, reading (a), window 11.
ca = CL[11][0]
print("\nunits per mouse in each cluster (reading (a), window 11):")
print(pd.crosstab(owner, ca).to_string())
fig, ax = plt.subplots(figsize=(7, 4))
order = np.argsort(ca, kind="stable")
im = ax.imshow(Xw[order], aspect="auto", cmap="RdBu_r", vmin=-5, vmax=5, interpolation="nearest")
ax.set_xticks([0, 50, 100, 150, 200, 250]); ax.set_xticklabels(["S -1", "S 0", "N -1", "N 0", "Sh -1", "Sh 0"])
ax.set_ylabel("unit (sorted by cluster)"); plt.colorbar(im, label="z (clipped at +/-5)"); plt.show()
print("plotted z from %.1f to %.1f; %.1f%% of values outside +/-5" % (Xw.min(), Xw.max(), 100 * np.mean(np.abs(Xw) > 5)))
# Clusters dominated by single mice, and the count without the sessions whose CS alignment is not
# shown (H22, H27, H34, H39; data check), for every window, reading (a).
for w in (11, 41):
    ct = pd.crosstab(owner, CL[w][0])
    share = (ct.max(axis=0) / ct.sum(axis=0)).round(2)
    print(f"window {w} bins, reading (a): largest share of one mouse in each cluster: {dict(zip(ct.columns, share))}; mouse: {dict(zip(ct.columns, ct.idxmax(axis=0)))}")
keep = ~np.isin(owner, ["H22", "H27", "H34", "H39"])
print("\nwithout H22, H27, H34 and H39 (%d units): clusters (a), (b) by window:" % keep.sum(),
      {w: tuple(int(c.max()) for c in cluster(F[keep], w)) for w in (1, 5, 11, 21, 31, 41, 51, 75, 101)})


**Result: blocked (methods_underspecified).** With the paper's rule (Ward linkage, correlation
distance, cut at 35% of the maximum linkage distance) the 1399 units of 24 mice give 8 to 12
clusters under reading (a) and 11 to 14 under reading (b) for smoothing windows of 20 to 420 ms,
and 6 to 8 under (a) for windows of 620 ms to 2 s; seven clusters come only under reading (a) with
an 820 ms window. The count depends on the smoothing window, which the paper sets by MATLAB's
"smoothing factor of 0.85" without the window it produced, so no reading can be preferred; the
count also moves with the cut fraction (14 at 25%, 10 at 35%, 7 at 40%, 5 at 50% for a 220 ms
window). Without H22, H27, H34 and H39, whose CS alignment is not shown, seven clusters come under
(a) at 420 ms and under (b) at 1020 ms and 2 s. Some clusters are single sessions (at 220 ms one
cluster is 93% H27), so a count here would not show functional types. The units are not the
paper's sorted neurons.

### fig4c-photoidentification

> BLA-NAc and BLA-CeA neurons are identified by short-latency responses to light at the end of each
> session. The Methods' rule (1 ms PSTH 50 ms around light onset, z-scored against the 50 ms before;
> excitation above z = 1.98 within 6 ms, none 7 to 50 ms after with a 6 ms moving average, none in
> the fake session); readings (a) all pulse onsets of the trains and the onset of the long light,
> (b) the 1 Hz pulses only. Reproduced when the counts are within 10% of 39 BLA-NAc and 32 BLA-CeA
> neurons under one reading.

No file says which projection (NAc or CeA) carries the Cav2-Cre in each mouse, so the two counts
cannot be split. Their sum can still be tested: both counts are within 10% only if the total is
within 63.9 to 78.1 (0.9 and 1.1 times 71). The fake-session test needs to know which of the two
light blocks is the fake one. The Methods put the fake session first ("a phototagging session was
conducted after a 'fake' phototagging session in which the patch cord was disconnected", Methods,
In vivo electrophysiology), so block 1 is taken as the fake one; the reverse is run as a reading,
and for H31 to H42, where one block comes before the task, either order per session is allowed
(the lowest total over those choices is printed). Light events of 5 and 10 ms are the pulses; the
files' 100 ms events are the only longer light, and their onsets are added in reading (a).

In [ ]:
tag_rows = []
for e in E:
    on, off = e["trials"]["phototagging"]
    dur = off - on
    g = np.argmax(np.diff(on))
    block = [np.arange(len(on)) <= g, np.arange(len(on)) > g]
    def pulses(b, reading):
        p = on[block[b] & (dur <= 0.011)]
        if reading == "a":
            return np.sort(np.r_[p, on[block[b] & (np.abs(dur - 0.1) < 0.01)]])
        return p[(np.r_[np.inf, np.diff(p)] > 0.9) & (np.r_[np.diff(p), np.inf] > 0.9)]
    base = U[U.subject == e["subject"]].baseline_hz.values
    r = {"subject": e["subject"], "units": len(e["spikes"])}
    for rd in ("a", "b"):
        P = [pulses(0, rd), pulses(1, rd)]
        r[f"{rd}_pulses"] = len(P[1])
        # nb.photoresponsive(sp, real, fake) is "responsive to real and not to fake"; computed once per block.
        resp = [np.array([nb.photoresponsive(sp, P[b])[0] for sp in e["spikes"]]) for b in (0, 1)]
        for name, (fk, rl) in {"fake1": (0, 1), "fake2": (1, 0)}.items():
            ok = resp[rl] & ~resp[fk]
            r[f"{rd}_{name}"] = int(ok.sum()); r[f"{rd}_{name}_0.3Hz"] = int((ok & (base >= 0.3)).sum())
        r[f"{rd}_resp_block1"] = int(resp[0].sum()); r[f"{rd}_resp_block2"] = int(resp[1].sum())
    tag_rows.append(r)
tg = pd.DataFrame(tag_rows)
print(tg.to_string(index=False))
late = tg.subject.isin(list(span.subject[span.light_before_task > 0]))
print("\nlight event durations (s) and counts, first session:", dict(zip(*np.unique(np.round(E[0]["trials"]["phototagging"][1] - E[0]["trials"]["phototagging"][0], 3), return_counts=True))))
print("\ntagged units in total (paper 39 + 32 = 71; both counts within 10% needs 63.9 to 78.1):")
for rd in ("a", "b"):
    for sfx in ("", "_0.3Hz"):
        f1, f2 = tg[f"{rd}_fake1{sfx}"], tg[f"{rd}_fake2{sfx}"]
        lowest = int(f1[~late].sum() + np.minimum(f1[late], f2[late]).sum())
        print(f"reading ({rd}){' baseline >= 0.3 Hz' if sfx else ''}: block 1 fake {f1.sum()}, block 2 fake {f2.sum()}, "
              f"block 1 fake with either order in H31-H42 (lowest) {lowest}; "
              f"responsive without the fake test: block 1 {tg[f'{rd}_resp_block1'].sum()}, block 2 {tg[f'{rd}_resp_block2'].sum()}")
print("units responsive to both blocks (reading a), share of those responsive to block 2: %.2f"
      % ((tg.a_resp_block2 - tg.a_fake1).sum() / tg.a_resp_block2.sum()))
print("mice with no tagged unit (reading a, block 1 fake):", list(tg.subject[tg.a_fake1 == 0]))


**Result: not reproduced.** With the fake-session test the rule tags 91 to 256 units in total,
depending on the reading of the pulses, which block is the fake one, and the 0.3 Hz exclusion: the
fewest are 91 (1 Hz pulses, either block order allowed in H31 to H42, baseline at least 0.3 Hz), and
with the Methods' order (block 1 fake) 146 and 133 for readings (a) and (b). All are above 78.1, so
whatever the projection of each mouse, the BLA-NAc and BLA-CeA counts cannot both be within 10% of
39 and 32. Without the fake test, 1243 to 1474 units respond; 88% of the units responsive to block 2
also respond to block 1, so the two blocks evoke much the same short-latency responses. The units
are not sorted single units, and the projection labels are missing, so a count matching the paper
was not expected; the outcome rests on the total alone.

## Behavior claim (Figure 5d-h)

### fig5dh-behavior-clusters

> Unsupervised trial-by-trial clustering of pose-derived behavior gives seven clusters for sucrose
> trials and five for shock trials. Eight measures for the 300 frames (20 s) after CS onset; tsfresh
> features (about 6000); PCA, t-SNE on the top 30 PCs, a kernel density over the t-SNE map (seaborn
> kdeplot), local maxima as watershed markers; sucrose and shock trials separately. t-SNE perplexity,
> kernel bandwidth and seed are not given: the defaults are used with several seeds. Reproduced
> when the default settings give 7 and 5 clusters for most seeds.

The trials come from the sessions where the fitted pose clock holds (data check), with the CS onsets
of the trial table mapped onto the pose clock. Choices the paper does not make, each shown:
tsfresh 0.16.0 does not install with this Python (3.13), so the current tsfresh with its default,
comprehensive feature set is used (6264 features per trial, against the paper's "~6000"); features
that are not finite or constant are dropped; the features are z-scored before PCA (a standard choice
for features on different scales; the raw features are run as a variant). "The defaults of
scikit-learn" are read two ways: the current defaults (PCA initialisation, automatic learning rate)
and those of the scikit-learn releases contemporary with tsfresh 0.16.0 (2020: random
initialisation, learning rate 200). The kernel density is seaborn's default (scipy gaussian_kde,
Scott's bandwidth, 200 x 200 grid, 3 bandwidths beyond the data), and each local maximum of the
grid is one watershed region.

In [ ]:
# Pose measures for the 300 frames after each sucrose and each shock CS, aligned sessions only.
BT = {"sucrose": [], "shock": []}; BW = {"sucrose": [], "shock": []}; lost = {"sucrose": 0, "shock": 0}
Ed = {e["subject"]: e for e in E}
for subj in pl.subject[pl.aligned]:
    d = PC[subj][1]
    for k in BT:
        on = Ed[subj]["trials"][nb.CS_TYPES[k]][0]
        X, kept = nb.behavior_trials(d["measures"], d["clock"], d["rate"], d["t0"], on)
        BT[k].append(X); BW[k] += [subj] * len(X); lost[k] += len(on) - len(kept)
BT = {k: np.concatenate(v) for k, v in BT.items()}; BW = {k: np.array(v) for k, v in BW.items()}
print({k: v.shape for k, v in BT.items()}, "trials dropped (frames beyond the pose):", lost)
print("trials per mouse:", pd.DataFrame({k: pd.Series(BW[k]).value_counts() for k in BW}).fillna(0).astype(int).T.to_string())
for k in BT:
    print(k, "measure ranges:", {m: (round(float(np.nanmin(BT[k][:, j])), 1), round(float(np.nanmax(BT[k][:, j])), 1)) for j, m in enumerate(nb.POSE_MEASURES)})


In [ ]:
import logging
from tsfresh import extract_features
from tsfresh.feature_extraction import ComprehensiveFCParameters
logging.getLogger("tsfresh").setLevel(logging.ERROR)
FEAT = {}
for k, X in BT.items():
    t = time.time()
    df = pd.DataFrame({"id": np.repeat(np.arange(len(X)), X.shape[2]), "time": np.tile(np.arange(X.shape[2]), len(X))})
    for j, m in enumerate(nb.POSE_MEASURES):
        df[m] = X[:, j, :].ravel()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        Fk = extract_features(df, column_id="id", column_sort="time", default_fc_parameters=ComprehensiveFCParameters(),
                              n_jobs=4, disable_progressbar=True)
    V = Fk.replace([np.inf, -np.inf], np.nan)
    V = V.loc[:, V.notna().all() & (V.std() > 0)]
    FEAT[k] = V
    print(f"{k}: {len(X)} trials, {Fk.shape[1]} features, {V.shape[1]} finite and not constant ({time.time() - t:.0f} s)")


In [ ]:
rows = []
EMB = {}
for k, V in FEAT.items():
    for scaling in ("z-scored", "raw"):
        M = ((V - V.mean()) / V.std()).values if scaling == "z-scored" else V.values
        pca = PCA(30, random_state=0).fit(M)
        PCs = pca.transform(M)
        for defaults, kw in (("current", {}), ("2020", {"init": "random", "learning_rate": 200.0})):
            counts, counts5 = [], []
            for seed in range(5):
                Y = TSNE(2, random_state=seed, **kw).fit_transform(PCs)
                n, D = nb.density_peaks(Y); counts.append(n); counts5.append(nb.density_peaks(Y, rel_floor=0.05)[0])
                EMB[(k, scaling, defaults, seed)] = Y
            rows.append({"trials": k, "features": scaling, "tsne_defaults": defaults, "var_30pcs": round(pca.explained_variance_ratio_.sum(), 3),
                         "clusters_by_seed": counts, "clusters_peaks_above_5pct": counts5,
                         "target": 7 if k == "sucrose" else 5, "seeds_at_target": sum(c == (7 if k == "sucrose" else 5) for c in counts)})
bc = pd.DataFrame(rows)
print(bc.to_string(index=False))
both = {(f, d): (bc[(bc.trials == "sucrose") & (bc.features == f) & (bc.tsne_defaults == d)].seeds_at_target.iloc[0] >= 3,
                 bc[(bc.trials == "shock") & (bc.features == f) & (bc.tsne_defaults == d)].seeds_at_target.iloc[0] >= 3)
        for f in ("z-scored", "raw") for d in ("current", "2020")}
print("\n(sucrose 7 in most seeds, shock 5 in most seeds) per reading:", both)
print("readings with both:", [r for r, v in both.items() if all(v)])
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, k in zip(axes, ("sucrose", "shock")):
    Y = EMB[(k, "z-scored", "2020", 0)]
    ax.scatter(Y[:, 0], Y[:, 1], s=3, c=pd.factorize(BW[k])[0], cmap="tab20")
    ax.set_title(f"{k} trials, t-SNE (z-scored features, 2020 defaults, seed 0)"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("colours are mice; all points are plotted")
print(f"\nnotebook ran in {(time.time() - started) / 60:.1f} min")


**Result: partially reproduced.** On 1319 sucrose and 688 shock trials from the 21 sessions with
an aligned pose clock, the cluster count depends on two choices the paper does not state. Read
literally (raw features, the installed scikit-learn's defaults) the criterion is met: 7 sucrose and 5
shock clusters for all 5 seeds. With the t-SNE defaults of the paper's time (2020: random
initialisation, learning rate 200), which the paper's tsfresh 0.16.0 dates, and the features
z-scored, sucrose trials give 7 clusters for 3 of 5 seeds (7, 7, 8, 8, 7) but shock trials give 5
for only 1 of 5 (4, 4, 4, 5, 3). The other two readings give 9 and 4 (z-scored, current defaults)
and 4 to 6 and 3 to 6 (raw, 2020 defaults). Without the features z-scored, the 30 PCs carry all the
variance of a few features with the largest scales. The reading adopted is z-scored features with
the 2020 defaults, the best supported (an assumption): the sucrose count holds and the shock count
does not. The trials rest on a pose clock fitted to the data.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig3k-nt-sensor-sucrose-increase | partially reproduced (p = 0.0046 with all trials, p = 0.093 with the first 16) |
| fig3l-nt-sensor-airpuff-decrease | not reproduced (right direction, p = 0.074 at best) |
| fig3-nt-sensor-early-dip | not reproduced (sucrose below zero, airpuff above) |
| fig4-recording-count | partially reproduced (31 mice against 30; 1902 or 1739 units against 685) |
| fig4c-photoidentification | not reproduced (91 to 256 tagged units against 71) |
| fig4d-seven-clusters | blocked, methods_underspecified (7 clusters only with an 820 ms smoothing window) |
| fig5dh-behavior-clusters | partially reproduced (7 sucrose clusters, 4 shock clusters in most seeds, adopted reading) |
| 13 claims needing group or projection labels | blocked, metadata_insufficient |
| 19 claims on experiments not in the dandiset | blocked, data_missing |

**How the analysis differs from the paper's.** The photometry cues are taken as CS-US trials with the
US 2 s after onset, since the files mark neither the US nor unpaired cues; the photometry z-score is
run with per-trial and session baselines; the clustering smoothing window is run over a range since
MATLAB's smoothing factor cannot be reproduced; the units are the files' units, not sorted single
units; the fake phototagging block is taken as the first one, as the Methods order them; the pose is
put on the trial clock by a linear map fitted to the port entries, in 21 of 29 sessions; tsfresh is
the current version, not 0.16.0, and the t-SNE defaults and feature scaling are run both ways.

**What was found about the dataset.** The files carry no CRISPR or control label and no tagged
projection, which most of the paper's electrophysiology findings need; the units are not sorted
single units (frequent sub-millisecond intervals, spikes shared between units, 64 units in most
sessions); H17's spike times stop at 249 s of a 12118 s recording; the pose has no timestamps and
its nominal 15 frames per second runs about 1.2% fast against the trial clock, by tens of seconds;
there are 31 mice where the paper has 30, and two of them may not be recorded on their last day; six
sessions have no neutral CS though the description says 2:1:1; the two light blocks are not labelled
fake and real, and both evoke short-latency responses in most units; the photometry series
interleaves three frame types at 60 Hz in one series, in one session the LED labels move by one frame
against the data, the cue events last 2 s where the Methods give 4 s, and no event marks the US or
unpaired cues. Only the NT sensor experiment of the photometry is deposited; the behavioral cohorts,
the axon-terminal and somatic photometry, the anatomy and the slice physiology are not.